# Caculate F1 score

In [1]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()
        
        binary_preds = (preds > 0.0).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1*100, 3),
                'Accuracy': round(acc*100, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
batch_size = None

# === 根目录 ===
#root_dir = "../saved_ablation/celeA_complex/DSCM_effectiveness_step100.0_scale2.5_NTIFalse"
root_dir = "../saved_benchmark_full/celebahq_simple/SD3_DSCM_effectiveness_step50.0_scale1.8_BlendFalse_reverse/"
# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in ["Smiling","Eyeglasses"] if os.path.isdir(os.path.join(root_dir, name))]


for target_attr in ["Smiling","Eyeglasses"]:
    print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # skip missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # skip if target attr not in result

        scores = results[target_attr]
        if batch_size is None:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
        else:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing 'None' F1-score on attribute 'Smiling' under different do(.) conditions:
do(Smiling)	Smiling: F1 = 92.905, Acc = 92.889
do(Eyeglasses)	Smiling: F1 = 98.204, Acc = 98.333
Comparing 'None' F1-score on attribute 'Eyeglasses' under different do(.) conditions:
do(Smiling)	Eyeglasses: F1 = 99.029, Acc = 99.889
do(Eyeglasses)	Eyeglasses: F1 = 97.094, Acc = 94.667


In [2]:

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)

    print('do {} IDP_l1 distance is {}'.format(do_attr,np.mean(loaded_results['IDP_l1'])))
    print('do {} IDP_lpips distance is {}'.format(do_attr,np.mean(loaded_results['IDP_lpips'])))
    
    print('do {} reverse_l1 distance is {}'.format(do_attr,np.mean(loaded_results['reverse_l1'])))
    print('do {} reverse_lpips distance is {}'.format(do_attr,np.mean(loaded_results['reverse_lpips'])))
    
    print('do {} compos_l1 distance is {}'.format(do_attr,np.mean(loaded_results['compos_l1'])))
    print('do {} compos_lpips distance is {}'.format(do_attr,np.mean(loaded_results['compos_lpips'])))

do Smiling IDP_l1 distance is 0.05980471521615982
do Smiling IDP_lpips distance is 0.03493079915642738
do Smiling reverse_l1 distance is 0.0896710604429245
do Smiling reverse_lpips distance is 0.05737127736210823
do Smiling compos_l1 distance is 0.004976388532668352
do Smiling compos_lpips distance is 0.00016161520034074783
do Eyeglasses IDP_l1 distance is 0.07455193996429443
do Eyeglasses IDP_lpips distance is 0.05996367707848549
do Eyeglasses reverse_l1 distance is 0.08604968339204788
do Eyeglasses reverse_lpips distance is 0.054246168583631516
do Eyeglasses compos_l1 distance is 0.004976388532668352
do Eyeglasses compos_lpips distance is 0.00016161520034074783


In [ ]:
#dict_keys(['predictions', 'targets'])
loaded_results.keys()
#dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])
loaded_results['predictions'].keys()

dict_keys(['predictions', 'targets'])

# Test the classifier

In [4]:
import random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score

import sys
sys.path.append("../../../")
sys.path.append("/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark")
sys.path.append('/home/jovyan/fcvm-data-volume/kzzr229/workspace/MCPL-diffuser')

from edit_modules.load_celebahq import CelebAHQ
from models.classifiers.celeba_classifier import CelebaClassifier,Celeba_anticausal_Classifier

# ------------------------------
# 1. Load test dataset
# ------------------------------
def dataset_load_path(data_root, dataset, split='train'):
    data_dir = data_root
    data = CelebAHQ(root=data_dir, split=split, transform=None, download=False)
    num_images = len(data)
    if 'simple' in dataset:
        selected_item = ['Smiling', 'Eyeglasses', 'Mouth_Slightly_Open', 'Male', 'Bald', 'Wearing_Lipstick', 'Wearing_Hat']
    elif 'complex' in dataset:
        raise NotImplementedError('complex dataset not handled here')
    else:
        raise AssertionError(f'no such {dataset} dataset')

    attribute_ids = [data.attr_names.index(attr) for attr in selected_item]
    metrics = {attr: torch.as_tensor(data.attr[:, attr_id], dtype=torch.float32)
               for attr, attr_id in zip(selected_item, attribute_ids)}
    attrs = torch.cat([metrics[attr].unsqueeze(1) for attr in selected_item], dim=1)
    possible_values = {attr: torch.unique(values, dim=0) for attr, values in metrics.items()}
    return data, attrs, num_images

data_root = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/datasets/'
dataset = 'simple'
data, _, num_images = dataset_load_path(data_root, dataset, split='test')
ori_data, imglabel, _ = dataset_load_path(data_root, dataset, split='test')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    #transforms.Normalize([0.5], [0.5]*3),
])

# Assign transform dynamically (CelebAHQ returns PIL Image by default)
data.transform = transform
test_loader = DataLoader(data, batch_size=32, shuffle=False, num_workers=4)

# ------------------------------
# 3. Load classifier
# ------------------------------
attr_name = "Smiling"
cond_name = "Mouth_Slightly_Open"
attr_name = 'Eyeglasses'
attr_index = data.attr_names.index(attr_name)
cond_index = data.attr_names.index(cond_name)

cls = Celeba_anticausal_Classifier(attr=attr_name, num_outputs=1,pretrain=False)
#ckpt_path = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed1/Smiling_classifier-epoch=35-val_metric=0.9880.ckpt'
ckpt_path = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed1/Eyeglasses_classifier-epoch=18-val_metric=0.9896.ckpt'

# cls = CelebaClassifier(attr=attr_name, num_outputs=1)
# ckpt_path = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed/Smiling_classifier-epoch=13.ckpt'
#ckpt_path = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed/Eyeglasses_classifier-epoch=14.ckpt'
state_dict = torch.load(ckpt_path, map_location='cuda')["state_dict"]
cls.load_state_dict(state_dict)
cls.to('cuda')
cls.eval()

# ------------------------------
# 4. Evaluate on test set
# ------------------------------
all_preds = []
all_labels = []
misclassified_ids = []

with torch.no_grad():
    sample_id = 0
    for images, attrs in test_loader:
        images = images.to('cuda')
        labels = attrs[:, attr_index].long().to('cuda')

        cond_labels = attrs[:, cond_index].long().to('cuda').unsqueeze(1)
        logits = cls(images)
        probs = torch.sigmoid(logits).squeeze(1)
        preds = (probs >= 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # track misclassified sample ids
        mismatch = preds != labels
        mis_ids_in_batch = torch.nonzero(mismatch).squeeze(1).cpu().tolist()

        # map local batch indices to global indices
        misclassified_ids.extend([sample_id + i for i in mis_ids_in_batch])
        sample_id += images.shape[0]

# ------------------------------
# 5. Compute metrics
# ------------------------------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"✅ Test Accuracy: {acc:.4f}")
print(f"✅ Test F1 Score: {f1:.4f}")
print(f"❌ Misclassified sample IDs: {misclassified_ids[:50]} ...")
print(f"Total misclassified samples: {len(misclassified_ids)} out of {len(all_labels)}")

/tmp/ipykernel_478/3533777761.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location='cuda')["state_dict"]


✅ Test Accuracy: 0.9993
✅ Test F1 Score: 0.9943
❌ Misclassified sample IDs: [1338, 2576] ...
Total misclassified samples: 2 out of 2824
